# HW2 Part 1: Data Cleaning & Validation

This notebook diagnoses the dirty `ohlc_hw` table, repairs what can be repaired with evidence, quarantines what cannot, and writes the result to a new table `ohlc_hw_cleaned`. The raw table is never modified.

Run it from this homework folder, with a local working copy of the course database at `data/data.db`.

Tables written to `data/data.db`:

| Table | Content |
|---|---|
| `ohlc_hw_cleaned` | Retained rows with repair flags |
| `ohlc_hw_quarantine` | Rows removed from analysis, with the reason |
| `ohlc_hw_quality_report` | Data Quality Report (diagnosis before repairs) |
| `ohlc_hw_ticker_audit` | Raw ticker label → normalized ticker |

In [1]:
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

DB_PATH = Path("data/data.db")
RAW_TABLE = "ohlc_hw"
REF_TABLE = "ohlc"  # supplied reference table: trading calendar and adjustment factors
CLEAN_TABLE = "ohlc_hw_cleaned"
QUARANTINE_TABLE = "ohlc_hw_quarantine"
REPORT_TABLE = "ohlc_hw_quality_report"
AUDIT_TABLE = "ohlc_hw_ticker_audit"

RAW_PRICES = ["open", "high", "low", "close"]
ALL_PRICES = RAW_PRICES + ["adj_close"]
VALUE_COLS = ["ts", "ticker", "open", "high", "low", "close", "adj_close", "volume"]

assert DB_PATH.exists(), f"Database not found at {DB_PATH.resolve()}; copy data.db from bCourses there."
conn = sqlite3.connect(DB_PATH)

## 1.1 Load and inspect the data

In [2]:
raw = pd.read_sql_query(f"SELECT * FROM {RAW_TABLE}", conn)
raw_fingerprint = (len(raw), pd.util.hash_pandas_object(raw, index=False).sum())  # to prove the raw table is untouched

raw.info()
raw.describe()

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 31.6 MB


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


Columns:

| Column | Meaning |
|---|---|
| `ts` | Observation (trading) date, stored as text `YYYY-MM-DD` |
| `ticker` | Stock identifier, as delivered by the vendor (may contain markup) |
| `open`, `high`, `low`, `close` | Raw daily prices |
| `adj_close` | Vendor close adjusted for dividends and splits |
| `volume` | Shares traded that day |

In [3]:
print(f"Rows: {len(raw):,}")
print(f"Date range: {raw['ts'].min()} to {raw['ts'].max()}")
print(f"Distinct raw ticker labels: {raw['ticker'].nunique()}")
raw["ticker"].drop_duplicates().head(8).tolist()

Rows: 397,711
Date range: 1962-01-02 to 2024-12-06
Distinct raw ticker labels: 150


['CAT',
 'CVX',
 'HD',
 'AAPL',
 'TRV',
 'JNJ',
 'DIS',
 '<a target="_blank"><span style="color: blue">CSCO</span></a>']

The 150 raw labels are not 150 stocks: most of them are wrapped in HTML markup. Tickers are normalized (markup stripped, whitespace trimmed, upper case) before any grouping or deduplication, and every original label is kept in an audit table.

In [4]:
def normalize_ticker(label: str) -> str:
    # Strip HTML tags, surrounding whitespace and case differences from a raw ticker label.
    return re.sub(r"<[^>]+>", "", str(label)).strip().upper()


df = raw.rename(columns={"ticker": "raw_ticker"})
df["ticker"] = df["raw_ticker"].map(normalize_ticker)

ticker_audit = (
    df.groupby(["raw_ticker", "ticker"]).size().rename("rows").reset_index()
    .assign(had_markup=lambda t: t["raw_ticker"] != t["ticker"])
)
tickers = sorted(df["ticker"].unique())
print(f"{len(tickers)} tickers after normalization: {', '.join(tickers)}")
print(f"Rows with a non-canonical label: {int((df['raw_ticker'] != df['ticker']).sum()):,}")
labels_per_ticker = ticker_audit.groupby("ticker")["raw_ticker"].count()
print(f"Raw labels per ticker: {labels_per_ticker.min()} to {labels_per_ticker.max()} (plain label + markup variants)")
ticker_audit[ticker_audit["ticker"] == "AAPL"]

30 tickers after normalization: AAPL, AMGN, AMZN, AXP, BA, CAT, CRM, CSCO, CVX, DIS, GS, HD, HON, IBM, JNJ, JPM, KO, MCD, MMM, MRK, MSFT, NKE, NVDA, PG, SHW, TRV, UNH, V, VZ, WMT
Rows with a non-canonical label: 39,936
Raw labels per ticker: 5 to 5 (plain label + markup variants)


,raw_ticker,ticker,rows,had_markup
0,"<a target=""_blank""><span style=""color: blue"">A...",AAPL,307,True
30,"<a target=""_blank""><span style=""color: red"">AA...",AAPL,318,True
60,"<a target=""_blank""><span style=""font-weight: b...",AAPL,306,True
90,"<a target=""_blank""><span style=""font-weight: b...",AAPL,264,True
120,AAPL,AAPL,10747,False


In [5]:
coverage = df.groupby("ticker")["ts"].agg(first_date="min", last_date="max", rows="count")
coverage

,first_date,last_date,rows
ticker,,,
AAPL,1980-12-12,2024-10-18,11942
AMGN,1983-06-17,2024-10-18,11290
AMZN,1997-05-15,2024-12-06,7458
AXP,1972-04-03,2024-10-18,14260
BA,1962-01-02,2024-10-18,16980
CAT,1962-01-02,2024-10-18,17044
CRM,2004-06-23,2024-10-18,5525
CSCO,1990-02-16,2024-10-18,9427
CVX,1962-01-02,2024-12-06,17073


## 1.2 Detect data quality issues

All detectors take a DataFrame with a normalized `ticker` column and return flags or issue rows; they do not modify the data. Duplicates are diagnosed on the full raw table. The other checks run on the table after exact duplicates are removed, so that one bad observation is not reported twice.

### Duplicates

In [6]:
def detect_duplicates(frame: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    # Return (exact_duplicate, conflicting_key) boolean masks.
    # exact_duplicate: the row repeats an earlier row on every value column (normalized ticker).
    # conflicting_key: after exact duplicates are dropped, several rows still share (ticker, ts).
    exact = frame.duplicated(subset=VALUE_COLS, keep="first")
    remaining = frame.loc[~exact]
    conflict = remaining.duplicated(subset=["ticker", "ts"], keep=False).reindex(frame.index, fill_value=False)
    return exact, conflict


exact_dup, conflict_key = detect_duplicates(df)
print(f"Exact duplicate rows: {int(exact_dup.sum()):,}")
print(f"Rows on conflicting (ticker, ts) keys: {int(conflict_key.sum()):,}")

dedup = df.loc[~exact_dup].sort_values(["ticker", "ts"]).reset_index(drop=True)
print(f"Rows after removing exact duplicates: {len(dedup):,}")

Exact duplicate rows: 29,460
Rows on conflicting (ticker, ts) keys: 0
Rows after removing exact duplicates: 368,251


### Missing data

In [7]:
def detect_missing_values(frame: pd.DataFrame) -> pd.DataFrame:
    # Count missing cells per column.
    out = frame[VALUE_COLS].isna().sum().rename("missing").to_frame()
    out["share"] = out["missing"] / len(frame)
    return out


detect_missing_values(dedup)

,missing,share
ts,0,0.000000
ticker,0,0.000000
open,0,0.000000
high,0,0.000000
low,0,0.000000
close,0,0.000000
adj_close,18374,0.049895
volume,0,0.000000


Missing trading days are a different check from missing cells. The reference calendar for each ticker is the set of dates of that ticker in the supplied `ohlc` table, restricted to the span of its history in `ohlc_hw`. Plain business days would wrongly count exchange holidays as gaps.

In [8]:
def detect_missing_trading_days(frame: pd.DataFrame, calendar: pd.DataFrame) -> pd.DataFrame:
    # Return one row per (ticker, ts) present in the reference calendar but absent from frame,
    # within each ticker's first-to-last observed date in frame.
    span = frame.groupby("ticker")["ts"].agg(first="min", last="max")
    cal = calendar.merge(span, left_on="ticker", right_index=True)
    cal = cal[(cal["ts"] >= cal["first"]) & (cal["ts"] <= cal["last"])]
    have = frame[["ticker", "ts"]].drop_duplicates()
    missing = cal.merge(have, on=["ticker", "ts"], how="left", indicator=True)
    return missing.loc[missing["_merge"] == "left_only", ["ticker", "ts"]].reset_index(drop=True)


def summarize_missing_days(missing: pd.DataFrame, frame: pd.DataFrame) -> list[str]:
    # One sentence per ticker, including tickers with zero gaps.
    span = frame.groupby("ticker")["ts"].agg(first="min", last="max")
    counts = missing.groupby("ticker").size()
    return [
        f"Ticker {t} is missing {int(counts.get(t, 0))} trading days between {row['first']} and {row['last']}"
        for t, row in span.iterrows()
    ]


calendar = pd.read_sql_query(f"SELECT DISTINCT ticker, ts FROM {REF_TABLE}", conn)
missing_days = detect_missing_trading_days(dedup, calendar)
print(f"Missing trading days in ohlc_hw: {len(missing_days)}")
print("\n".join(summarize_missing_days(missing_days, dedup)[:5]), "\n...")

Missing trading days in ohlc_hw: 0
Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18 
...


The real table has no missing trading days. To show that the detector works, we remove one date from a small slice and check that it is found.

In [9]:
demo = dedup[(dedup["ticker"] == "AAPL") & (dedup["ts"].between("2024-03-01", "2024-03-15"))]
removed_date = demo["ts"].iloc[4]
demo_gap = demo[demo["ts"] != removed_date]
found = detect_missing_trading_days(demo_gap, calendar)
print(f"Removed {removed_date}; detector found: {found['ts'].tolist()}")
assert found["ts"].tolist() == [removed_date]

Removed 2024-03-07; detector found: ['2024-03-07']


### Outliers and impossible prices

Daily simple returns on raw close are computed within each ticker after sorting by date. A return is flagged when it is more than 3 sample standard deviations (`ddof=1`) from the ticker's mean return, in either tail. Undefined returns (first observation, missing close) are excluded, and a ticker with zero or undefined standard deviation gets no flags. This is a review flag, not a deletion rule.

In [10]:
def add_simple_returns(frame: pd.DataFrame) -> pd.DataFrame:
    # Add the daily simple return (close - prev_close) / prev_close within each ticker.
    out = frame.sort_values(["ticker", "ts"]).copy()
    out["ret"] = out.groupby("ticker")["close"].pct_change(fill_method=None)
    return out


def detect_return_outliers(frame: pd.DataFrame, n_std: float = 3.0) -> pd.Series:
    # Flag |ret - mean| > n_std * std within each ticker (sample std, both tails).
    grouped = frame.groupby("ticker")["ret"]
    mean = grouped.transform("mean")
    std = grouped.transform(lambda s: s.std(ddof=1))
    valid = frame["ret"].notna() & np.isfinite(frame["ret"]) & std.gt(0)
    return valid & ((frame["ret"] - mean).abs() > n_std * std)


def detect_price_inconsistencies(frame: pd.DataFrame) -> pd.DataFrame:
    # Boolean flags for impossible or invalid prices and suspicious volume.
    p = frame[ALL_PRICES]
    flags = pd.DataFrame(index=frame.index)
    flags["high_below_close"] = frame["high"] < frame["close"]
    flags["low_above_close"] = frame["low"] > frame["close"]
    flags["open_outside_range"] = (frame["open"] < frame["low"]) | (frame["open"] > frame["high"])
    flags["high_below_low"] = frame["high"] < frame["low"]
    flags["nonpositive_raw_price"] = (p[RAW_PRICES] <= 0).any(axis=1)
    flags["nonfinite_raw_price"] = np.isinf(p[RAW_PRICES]).any(axis=1)
    flags["nonpositive_adj_close"] = p["adj_close"] <= 0
    flags["nonfinite_adj_close"] = np.isinf(p["adj_close"])
    flags["zero_volume"] = frame["volume"] == 0
    flags["negative_volume"] = frame["volume"] < 0
    return flags


dedup = add_simple_returns(dedup)
dedup["return_outlier"] = detect_return_outliers(dedup)
price_flags = detect_price_inconsistencies(dedup)
print(f"Return outliers (3 std): {int(dedup['return_outlier'].sum()):,}")
price_flags.sum().to_frame("rows")

Return outliers (3 std): 5,056


,rows
high_below_close,0
low_above_close,0
open_outside_range,5
high_below_low,0
nonpositive_raw_price,5
nonfinite_raw_price,0
nonpositive_adj_close,0
nonfinite_adj_close,0
zero_volume,75
negative_volume,0


The raw OHLC bounds are respected everywhere except five rows whose `open` is 0, which is below `low`. Raw bounds do not apply to `adj_close`, but `adj_close` has its own problem: comparing it with raw close shows isolated values that are several times too large.

The ratio `adj_close / close` is the adjustment factor. It changes slowly (dividends) or by a split ratio on a split date, and it never jumps up for one day and comes back. Each factor is compared with the median factor of the surrounding 11 observations of the same ticker; a ratio above 2 or below 1/2 marks the adjusted close as corrupted.

In [11]:
def adjustment_factor(frame: pd.DataFrame) -> pd.Series:
    return frame["adj_close"] / frame["close"]


def detect_adj_factor_anomalies(frame: pd.DataFrame, window: int = 11, threshold: float = 2.0) -> pd.Series:
    # Flag adjusted closes whose factor departs from the local median factor by more than threshold.
    f = adjustment_factor(frame)
    local = f.groupby(frame["ticker"]).transform(
        lambda s: s.rolling(window, center=True, min_periods=3).median()
    )
    ratio = f / local
    return (ratio > threshold) | (ratio < 1 / threshold)


dedup["adj_corrupted"] = detect_adj_factor_anomalies(dedup)
factor_ratio = (adjustment_factor(dedup) / adjustment_factor(dedup).groupby(dedup["ticker"]).transform(
    lambda s: s.rolling(11, center=True, min_periods=3).median()))
print(f"Corrupted adjusted closes: {int(dedup['adj_corrupted'].sum()):,}")
print(f"Factor ratio of flagged rows: min {factor_ratio[dedup['adj_corrupted']].min():.1f}, "
      f"max {factor_ratio[dedup['adj_corrupted']].max():.1f}")
print(f"Largest ratio among unflagged rows: {factor_ratio[~dedup['adj_corrupted']].max():.3f}")

Corrupted adjusted closes: 10,417


Factor ratio of flagged rows: min 4.0, max 24.0
Largest ratio among unflagged rows: 1.523


The flagged rows have factors 4 to 24 times their local median, while no other row goes above 1.53: the threshold of 2 sits in a wide gap between the two groups, so it is not a fine-tuned choice.

#### Investigating the extreme returns

The raw `close` series is already split-adjusted by the vendor (for example, AAPL shows no jump at its August 2020 4:1 split), so a 3-std move is not a split by default. Each flagged return is classified with explicit rules:

- Spike and reversal: a move above 25% that is undone the next day (the two-day product returns within 5% of the starting level). A one-day round trip of that size is a bad print, and the next-day reversal belongs to the same error.
- Suspected unadjusted split: a move above 45% that is not reversed, and whose price ratio is within 2% of a common split ratio (2, 3, 4, 5, 10 or their inverses).
- Otherwise the move is retained and flagged. When at least 5 tickers are flagged on the same date, the description records it as a market-wide move.

In [12]:
SPLIT_RATIOS = np.array([2, 3, 4, 5, 10], dtype=float)


def classify_extreme_returns(frame: pd.DataFrame) -> pd.DataFrame:
    # Return one row per flagged return with a classification and evidence.
    f = frame.copy()
    g = f.groupby("ticker")
    f["next_ret"] = g["ret"].shift(-1)
    f["prev_ret"] = g["ret"].shift(1)
    round_trip = (1 + f["ret"]) * (1 + f["next_ret"]) - 1
    f["spike"] = (f["ret"].abs() > 0.25) & (f["ret"] * f["next_ret"] < 0) & (round_trip.abs() < 0.05)
    f["spike_reversal"] = g["spike"].shift(1, fill_value=False)

    ratio = 1 + f["ret"]
    candidates = np.concatenate([SPLIT_RATIOS, 1 / SPLIT_RATIOS])
    big = f["ret"].abs() > 0.45
    nearest = ratio[big].apply(lambda r: candidates[np.argmin(np.abs(candidates / r - 1))])
    f["split_like"] = big & ~f["spike"] & ((nearest / ratio[big] - 1).abs() < 0.02).reindex(f.index, fill_value=False)

    flagged = f[f["return_outlier"]].copy()
    same_day = flagged.groupby("ts")["ticker"].transform("count")

    def describe(row, n_same_day):
        if row["spike"]:
            return ("spike_error", "high",
                    f"Close {row['ret']:+.1%} then {row['next_ret']:+.1%} next day: one-day round trip "
                    f"(O=H=L=C: {row['open'] == row['high'] == row['low'] == row['close']}, volume {row['volume']:,}); confirmed bad print")
        if row["spike_reversal"]:
            return ("spike_error", "high",
                    f"Close {row['ret']:+.1%}: reversal of the previous day's bad print")
        if row["split_like"]:
            return ("suspected_unadjusted_split", "high",
                    f"Close {row['ret']:+.1%} (price ratio {1 + row['ret']:.3f}) with no reversal, close to a split ratio; "
                    "the vendor adjusted close shows the same jump. Cause not verified")
        note = f"; market-wide: {n_same_day} tickers flagged on this date" if n_same_day >= 5 else ""
        return ("extreme_return", "low",
                f"Close {row['ret']:+.1%}, beyond 3 std of the ticker's daily returns; OHLC consistent, "
                f"no reversal; retained as a genuine move (cause not verified){note}")

    out = [(r["ticker"], *describe(r, n), r["ts"], bool(r["spike"])) for (_, r), n in zip(flagged.iterrows(), same_day)]
    # bad_print marks the first day of a spike (the value to repair); it is not a report column
    return pd.DataFrame(out, columns=["ticker", "issue_type", "severity", "description", "issue_date", "bad_print"])


extreme = classify_extreme_returns(dedup)
extreme.groupby(["issue_type", "severity"]).size().to_frame("rows")

,,rows
issue_type,severity,
extreme_return,low,5049
spike_error,high,6
suspected_unadjusted_split,high,1


In [13]:
extreme[extreme["issue_type"] != "extreme_return"]

,ticker,issue_type,severity,description,issue_date,bad_print
3066,MCD,suspected_unadjusted_split,high,Close -75.4% (price ratio 0.246) with no rever...,1968-01-02,False
4425,TRV,spike_error,high,Close +100.0% then -49.1% next day: one-day ro...,1976-10-28,True
4426,TRV,spike_error,high,Close -49.1%: reversal of the previous day's b...,1976-10-29,False
4427,TRV,spike_error,high,Close +99.8% then -49.1% next day: one-day rou...,1978-12-20,True
4428,TRV,spike_error,high,Close -49.1%: reversal of the previous day's b...,1978-12-21,False
4429,TRV,spike_error,high,Close +100.0% then -50.0% next day: one-day ro...,1981-08-10,True
4430,TRV,spike_error,high,Close -50.0%: reversal of the previous day's b...,1981-08-11,False


The three TRV spikes are flat bars (open = high = low = close) at about twice the surrounding price, followed by a return to the previous level, and one of them has zero volume: these are bad prints.

The MCD move on 1968-01-02 is a permanent drop to 0.246 of the previous close, close to 1/4, at the very start of its history; the vendor's adjusted close drops by the same amount. This looks like a 4:1 split that was never applied to the older prices, but there is no corporate-action source here to confirm it.

In [14]:
cols = ["ticker", "ts", "open", "high", "low", "close", "volume", "ret"]
pd.concat([
    dedup[(dedup["ticker"] == "TRV") & dedup["ts"].between("1976-10-26", "1976-11-01")][cols],
    dedup[(dedup["ticker"] == "MCD") & dedup["ts"].between("1967-12-27", "1968-01-04")][cols],
])

,ticker,ts,open,high,low,close,volume,ret
318330,TRV,1976-10-26,4.50000,4.50000,4.50000,4.50000,91200,-0.004425
318331,TRV,1976-10-27,4.61000,4.61000,4.61000,4.61000,230400,0.024444
318332,TRV,1976-10-28,9.22000,9.22000,9.22000,9.22000,0,1.000000
318333,TRV,1976-10-29,4.69000,4.69000,4.69000,4.69000,79200,-0.491323
318334,TRV,1976-11-01,4.66000,4.66000,4.66000,4.66000,59200,-0.006397
213785,MCD,1967-12-27,0.74074,0.74074,0.72634,0.74074,498150,0.000000
213786,MCD,1967-12-28,0.74074,0.74897,0.73868,0.73868,194400,-0.002781
213787,MCD,1967-12-29,0.73868,0.74074,0.70988,0.70988,1093500,-0.038988
213788,MCD,1968-01-02,0.17490,0.17490,0.17490,0.17490,6755143,-0.753620
213789,MCD,1968-01-03,0.17567,0.17567,0.17567,0.17567,3839207,0.004403


### Data Quality Report

One row per issue and per observation, so an observation can appear several times. Label and duplicate issues are diagnosed on the raw table; all other issues on the deduplicated table, before any repair.

Severity:

| Severity | Meaning | Issue types |
|---|---|---|
| high | Would distort prices or returns if left in place | corrupted `adj_close`, impossible prices, nonpositive or non-finite prices, spike errors, suspected unadjusted split, conflicting keys |
| medium | Missing or doubtful information that is repairable | missing `adj_close`, missing raw price or volume, missing trading day, zero volume |
| low | Cosmetic or review-only | ticker markup, exact duplicate, genuine extreme return |

In [15]:
def issue_rows(frame: pd.DataFrame, mask: pd.Series, issue_type: str, severity: str, description) -> pd.DataFrame:
    # Build report rows for the observations selected by mask.
    # description is a fixed string or a function mapping the selected rows to a Series of strings.
    sel = frame.loc[mask]
    desc = description(sel) if callable(description) else description
    return pd.DataFrame({"ticker": sel["ticker"], "issue_type": issue_type, "issue_date": sel["ts"],
                         "description": desc, "severity": severity})


def ohlc_text(sel: pd.DataFrame) -> pd.Series:
    return ("O=" + sel["open"].astype(str) + ", H=" + sel["high"].astype(str)
            + ", L=" + sel["low"].astype(str) + ", C=" + sel["close"].astype(str))


def build_quality_report(raw_frame, dedup_frame, flags, exact, conflict, missing, extreme_rows) -> pd.DataFrame:
    parts = [
        issue_rows(raw_frame, raw_frame["raw_ticker"] != raw_frame["ticker"], "ticker_markup", "low",
                   lambda s: "Raw label " + s["raw_ticker"] + " normalized to " + s["ticker"]),
        issue_rows(raw_frame, exact, "exact_duplicate", "low", "Repeats an earlier record on every column"),
        issue_rows(raw_frame, conflict, "conflicting_key", "high", "Several different rows for the same (ticker, ts)"),
    ]
    for col in RAW_PRICES + ["volume"]:
        parts.append(issue_rows(dedup_frame, dedup_frame[col].isna(), f"missing_{col}", "medium", f"{col} is missing"))
    parts.append(issue_rows(dedup_frame, dedup_frame["adj_close"].isna(), "missing_adj_close", "medium",
                            "adj_close is missing"))
    parts.append(issue_rows(dedup_frame, dedup_frame["adj_corrupted"], "corrupted_adj_close", "high",
                            lambda s: "adj_close " + s["adj_close"].round(4).astype(str) + " gives a factor "
                                      + (s["adj_close"] / s["close"]).round(3).astype(str)
                                      + ", far from the neighboring factors"))
    checks = {
        "high_below_close": "high < close", "low_above_close": "low > close",
        "open_outside_range": "open outside [low, high]", "high_below_low": "high < low",
        "nonpositive_raw_price": "raw price <= 0", "nonfinite_raw_price": "raw price not finite",
        "nonpositive_adj_close": "adj_close <= 0", "nonfinite_adj_close": "adj_close not finite",
        "negative_volume": "volume < 0",
    }
    for name, text in checks.items():
        parts.append(issue_rows(dedup_frame, flags[name], name, "high",
                                lambda s, t=text: t + " (" + ohlc_text(s) + ")"))
    flat = (dedup_frame["open"] == dedup_frame["close"]) & (dedup_frame["high"] == dedup_frame["low"])
    parts.append(issue_rows(dedup_frame, flags["zero_volume"], "zero_volume", "medium",
                            lambda s: np.where(flat[s.index], "Zero volume on a flat bar: possible stale record",
                                               "Zero volume: possible stale record")))
    parts.append(pd.DataFrame({"ticker": missing["ticker"], "issue_type": "missing_trading_day",
                               "issue_date": missing["ts"], "description": "Date in the reference calendar but absent",
                               "severity": "medium"}))
    parts.append(extreme_rows)
    report = pd.concat([p for p in parts if len(p)], ignore_index=True)
    return report[["ticker", "issue_type", "issue_date", "description", "severity"]].sort_values(
        ["ticker", "issue_date", "issue_type"]).reset_index(drop=True)


quality_report = build_quality_report(df, dedup, price_flags, exact_dup, conflict_key, missing_days, extreme)
print(f"Data Quality Report: {len(quality_report):,} issue rows")
quality_report.groupby(["severity", "issue_type"]).size().rename("rows").to_frame()

Data Quality Report: 103,328 issue rows


rows
severity issue_type                       
high     corrupted_adj_close         10417
         nonpositive_raw_price           5
         open_outside_range              5
         spike_error                     6
         suspected_unadjusted_split      1
low      exact_duplicate             29460
         extreme_return               5049
         ticker_markup               39936
medium   missing_adj_close           18374
         zero_volume                    75

In [16]:
quality_report[quality_report["severity"] == "high"].drop_duplicates("issue_type")

,ticker,issue_type,issue_date,description,severity
8,AAPL,corrupted_adj_close,1981-02-03,"adj_close 1.2646 gives a factor 10.254, far fr...",high
24428,CVX,nonpositive_raw_price,1962-01-02,"raw price <= 0 (O=0.0, H=3.3, L=3.24, C=3.3)",high
24429,CVX,open_outside_range,1962-01-02,"open outside [low, high] (O=0.0, H=3.3, L=3.24...",high
60116,MCD,suspected_unadjusted_split,1968-01-02,Close -75.4% (price ratio 0.246) with no rever...,high
89374,TRV,spike_error,1976-10-28,Close +100.0% then -49.1% next day: one-day ro...,high


## 1.3 Clean the data

Repairs run in a fixed order, within each ticker sorted by date, and the OHLC validation is rerun after every step.

1. Remove exact duplicates; quarantine both rows of a conflicting key, since there is no basis to pick one.
2. Quarantine rows where all four raw prices are missing, before any filling.
3. Forward-fill missing raw prices within the ticker and flag them; leading gaps cannot be filled and are quarantined.
4. Fill missing volume with 0 and flag it: unknown volume is not evidence of no trading.
5. Impossible prices (Option 1): the offending field is set to NaN and forward-filled from the same ticker. If the result still breaks the day's range, or there is no earlier value, the row is quarantined.
6. Confirmed spike errors: the close is linearly interpolated between the two neighboring days; a flat bar gets the same value for open, high and low.
7. Suspected unadjusted split: the earlier block of that ticker is quarantined, because the later block is the one continuous with today's prices.
8. Adjusted close: missing or corrupted values are rebuilt as validated raw close times a date-specific factor `adj_close / close` taken from the matching `(ticker, ts)` row of the supplied `ohlc` table. For a repaired close, the factor is interpolated between the neighbors instead. Factors are never forward-filled across dates.

In [17]:
def validate_ohlc(frame: pd.DataFrame) -> pd.Series:
    # Count violations of the raw price rules and adjusted-price consistency.
    p = frame[RAW_PRICES]
    f = adjustment_factor(frame)
    step = (f / f.groupby(frame["ticker"]).shift(1)).abs()
    return pd.Series({
        "missing_raw_price": int(p.isna().any(axis=1).sum()),
        "nonpositive_or_nonfinite_raw": int(((p <= 0) | np.isinf(p)).any(axis=1).sum()),
        "close_outside_range": int(((frame["close"] > frame["high"]) | (frame["close"] < frame["low"])).sum()),
        "open_outside_range": int(((frame["open"] > frame["high"]) | (frame["open"] < frame["low"])).sum()),
        "missing_adj_close": int(frame["adj_close"].isna().sum()),
        "nonpositive_or_nonfinite_adj": int(((frame["adj_close"] <= 0) | np.isinf(frame["adj_close"])).sum()),
        "adj_factor_jump_over_50pct": int(((step > 1.5) | (step < 1 / 1.5)).sum()),
        "duplicate_keys": int(frame.duplicated(["ticker", "ts"]).sum()),
    })


class Cleaner:
    # Apply the repair steps in order, keeping a quarantine and a validation log.

    def __init__(self, frame: pd.DataFrame, reference: pd.DataFrame | None):
        self.df = frame.sort_values(["ticker", "ts"]).reset_index(drop=True)
        for col in ["price_filled", "price_interpolated", "volume_filled"]:
            self.df[col] = False
        self.df["adj_source"] = np.where(self.df["adj_close"].notna(), "vendor", "missing")
        self.reference = reference
        self.quarantine = []
        self.log = [("start", len(self.df), validate_ohlc(self.df))]

    def _record(self, step):
        self.log.append((step, len(self.df), validate_ohlc(self.df)))

    def _quarantine(self, mask: pd.Series, reason: str):
        if mask.any():
            self.quarantine.append(self.df.loc[mask].assign(reason=reason))
            self.df = self.df.loc[~mask].reset_index(drop=True)

    def remove_duplicates(self):
        exact = self.df.duplicated(subset=VALUE_COLS, keep="first")
        self.n_exact_duplicates = int(exact.sum())
        self.df = self.df.loc[~exact].reset_index(drop=True)
        conflict = self.df.duplicated(["ticker", "ts"], keep=False)
        self._quarantine(conflict, "conflicting (ticker, ts) key")
        self._record("1. duplicates")

    def drop_all_prices_missing(self):
        self._quarantine(self.df[RAW_PRICES].isna().all(axis=1), "all raw prices missing")
        self._record("2. all-missing rows")

    def fill_missing_prices(self):
        missing = self.df[RAW_PRICES].isna()
        filled = self.df.groupby("ticker")[RAW_PRICES].ffill()
        self.df[RAW_PRICES] = filled
        self.df["price_filled"] |= missing.any(axis=1) & filled.notna().all(axis=1)
        self._quarantine(self.df[RAW_PRICES].isna().any(axis=1), "leading missing price (nothing to carry forward)")
        self._record("3. forward-fill prices")

    def fill_missing_volume(self):
        missing = self.df["volume"].isna()
        self.df.loc[missing, "volume"] = 0
        self.df["volume_filled"] |= missing
        self._record("4. fill volume")

    def repair_impossible_prices(self):
        # Option 1: set the offending open/close to NaN, forward-fill it within the ticker, recheck the range.
        d = self.df
        bad_hl = ((d["high"] < d["low"]) | (d[["high", "low"]] <= 0).any(axis=1)
                  | ~np.isfinite(d[["high", "low"]]).all(axis=1))
        self._quarantine(bad_hl, "high/low range invalid")
        d = self.df

        def out_of_range(col):
            return (d[col] <= 0) | ~np.isfinite(d[col]) | (d[col] < d["low"]) | (d[col] > d["high"])

        bad_open, bad_close = out_of_range("open"), out_of_range("close")
        touched = bad_open | bad_close
        d.loc[bad_open, "open"] = np.nan
        d.loc[bad_close, "close"] = np.nan
        d[["open", "close"]] = d.groupby("ticker")[["open", "close"]].ffill()
        still_bad = touched & (out_of_range("open") | out_of_range("close"))
        d["price_filled"] |= touched & ~still_bad
        self.df = d
        self._quarantine(still_bad, "impossible price, no valid replacement within the day's range")
        self._record("5. impossible prices")

    def repair_spikes(self, spikes: pd.DataFrame):
        # spikes: (ticker, ts) of confirmed one-day bad prints.
        d = self.df
        key = d.set_index(["ticker", "ts"]).index
        hit = key.isin(pd.MultiIndex.from_frame(spikes[["ticker", "ts"]]))
        flat = hit & (d["open"] == d["close"]) & (d["high"] == d["close"]) & (d["low"] == d["close"])
        d.loc[hit, "close"] = np.nan
        d["close"] = d.groupby("ticker")["close"].transform(lambda s: s.interpolate(limit_area="inside"))
        for col in ["open", "high", "low"]:
            d.loc[flat, col] = d.loc[flat, "close"]
        d.loc[hit, "price_interpolated"] = True
        d.loc[hit, "adj_close"] = np.nan  # the vendor value refers to the bad close
        d.loc[hit, "adj_source"] = "missing"
        self.df = d
        bad = pd.Series(hit, index=d.index) & (d["close"].isna() | (d["close"] > d["high"]) | (d["close"] < d["low"]))
        self._quarantine(bad, "bad print that cannot be interpolated inside the day's range")
        self._record("6. spike errors")

    def quarantine_before(self, ticker: str, date: str, reason: str):
        self._quarantine((self.df["ticker"] == ticker) & (self.df["ts"] < date), reason)
        self._record(f"7. {ticker} before {date}")

    def repair_adjusted_close(self, corrupted: pd.Series):
        # corrupted: boolean Series aligned on self.df marking vendor adj_close values to discard.
        d = self.df
        d.loc[corrupted, "adj_close"] = np.nan
        need = d["adj_close"].isna()
        if self.reference is not None:
            ref = self.reference.rename(columns={"close": "ref_close", "adj_close": "ref_adj"})
            m = d[["ticker", "ts", "close"]].merge(ref, on=["ticker", "ts"], how="left")
            ref_factor = (m["ref_adj"] / m["ref_close"]).set_axis(d.index)
            # use the reference factor only where our raw close is the validated vendor close
            same_close = pd.Series(np.isclose(m["close"], m["ref_close"]), index=d.index)
            usable = need & ~d["price_interpolated"] & same_close & ref_factor.gt(0)
            d.loc[usable, "adj_close"] = d.loc[usable, "close"] * ref_factor[usable]
            d.loc[usable, "adj_source"] = "reference_factor"
        # repaired closes: interpolate the factor between neighbors that agree within 1%
        f = adjustment_factor(d)
        for idx in d.index[d["adj_close"].isna() & d["price_interpolated"]]:
            same = d["ticker"] == d.at[idx, "ticker"]
            before = f[same & (d.index < idx)].dropna()
            after = f[same & (d.index > idx)].dropna()
            if len(before) and len(after) and abs(after.iloc[0] / before.iloc[-1] - 1) < 0.01:
                d.at[idx, "adj_close"] = d.at[idx, "close"] * (before.iloc[-1] + after.iloc[0]) / 2
                d.at[idx, "adj_source"] = "interpolated_factor"
        self.df = d
        self._quarantine(d["adj_close"].isna(), "adj_close unresolved (no trusted factor)")
        self._record("8. adjusted close")

    def validation_log(self) -> pd.DataFrame:
        return pd.DataFrame({step: v for step, _, v in self.log}).T.assign(
            rows=[n for _, n, _ in self.log])

    def quarantined(self) -> pd.DataFrame:
        if not self.quarantine:
            return pd.DataFrame(columns=list(self.df.columns) + ["reason"])
        return pd.concat(self.quarantine, ignore_index=True)

### Demonstration on injected faults

The real table has no missing raw prices, missing volumes, conflicting keys, `high < close` or `low > close` rows. The same detectors and repairs are first run on a small hand-made table that contains each of these faults.

In [18]:
toy = pd.DataFrame({
    "ts":       ["2024-01-02", "2024-01-03", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-05",
                 "2024-01-08", "2024-01-09", "2024-01-02", "2024-01-03"],
    "ticker":   ["AAA", "AAA", "AAA", "AAA", "AAA", "AAA", "AAA", "AAA", "BBB", "BBB"],
    "open":     [10.0, 10.2, 10.2, np.nan, 10.4, 10.9, np.nan, 10.6, np.nan, 20.0],
    "high":     [10.5, 10.6, 10.6, 10.7, 10.8, 11.0, np.nan, 10.9, np.nan, 20.5],
    "low":      [9.8, 10.0, 10.0, 10.1, 10.2, 10.5, np.nan, 10.4, np.nan, 19.5],
    "close":    [10.3, 10.4, 10.4, 10.5, 10.6, 10.8, np.nan, 11.2, 19.9, 20.1],
    "adj_close":[10.3, 10.4, 10.4, 10.5, 10.6, 10.8, np.nan, 11.2, 19.9, 20.1],
    "volume":   [100, 120, 120, np.nan, 90, 95, 80, 110, 50, 60],
})
toy["raw_ticker"] = toy["ticker"]
# rows: exact duplicate (01-03), missing open/volume (01-04), conflicting key (01-05),
# all prices missing (01-08), close above high (01-09), leading missing prices (BBB 01-02)
toy_exact, toy_conflict = detect_duplicates(toy)
print("exact duplicates:", toy.loc[toy_exact, "ts"].tolist(), "| conflicting keys:", toy.loc[toy_conflict, "ts"].tolist())
print("missing cells:", detect_missing_values(toy)["missing"][lambda s: s > 0].to_dict())
print("price flags:", detect_price_inconsistencies(toy).sum()[lambda s: s > 0].to_dict())

toy_clean = Cleaner(toy, reference=None)
toy_clean.remove_duplicates()
toy_clean.drop_all_prices_missing()
toy_clean.fill_missing_prices()
toy_clean.fill_missing_volume()
toy_clean.repair_impossible_prices()
toy_clean.df

exact duplicates: ['2024-01-03'] | conflicting keys: ['2024-01-05', '2024-01-05']
missing cells: {'open': 3, 'high': 2, 'low': 2, 'close': 1, 'adj_close': 1, 'volume': 1}
price flags: {'high_below_close': 1}


,ts,ticker,open,high,low,close,adj_close,volume,raw_ticker,price_filled,price_interpolated,volume_filled,adj_source
0,2024-01-02,AAA,10.0,10.5,9.8,10.3,10.3,100.0,AAA,False,False,False,vendor
1,2024-01-03,AAA,10.2,10.6,10.0,10.4,10.4,120.0,AAA,False,False,False,vendor
2,2024-01-04,AAA,10.2,10.7,10.1,10.5,10.5,0.0,AAA,True,False,True,vendor
3,2024-01-09,AAA,10.6,10.9,10.4,10.5,11.2,110.0,AAA,True,False,False,vendor
4,2024-01-03,BBB,20.0,20.5,19.5,20.1,20.1,60.0,BBB,False,False,False,vendor


In [19]:
toy_clean.quarantined()[["ticker", "ts", "open", "high", "low", "close", "reason"]]

,ticker,ts,open,high,low,close,reason
0,AAA,2024-01-05,10.4,10.8,10.2,10.6,"conflicting (ticker, ts) key"
1,AAA,2024-01-05,10.9,11.0,10.5,10.8,"conflicting (ticker, ts) key"
2,AAA,2024-01-08,NaN,NaN,NaN,NaN,all raw prices missing
3,BBB,2024-01-02,NaN,NaN,NaN,19.9,leading missing price (nothing to carry forward)


The duplicate is removed, both conflicting rows and the all-missing row are quarantined, the missing open is carried forward and flagged, the missing volume becomes 0 with a flag, and BBB's leading gap is quarantined rather than borrowed from another ticker. The 01-09 close above the high is replaced by the last valid close, 10.5, which lies inside that day's range, so the row is kept and flagged.

### Cleaning the real table

In [20]:
reference = pd.read_sql_query(f"SELECT ticker, ts, close, adj_close FROM {REF_TABLE}", conn)

cleaner = Cleaner(df, reference)
cleaner.remove_duplicates()
cleaner.drop_all_prices_missing()
cleaner.fill_missing_prices()
cleaner.fill_missing_volume()
cleaner.repair_impossible_prices()

# only the first day of each round trip is a bad print; the reversal day is a valid price
spikes = extreme.loc[extreme["bad_print"], ["ticker", "issue_date"]].rename(columns={"issue_date": "ts"})
cleaner.repair_spikes(spikes)

for _, row in extreme[extreme["issue_type"] == "suspected_unadjusted_split"].iterrows():
    cleaner.quarantine_before(row["ticker"], row["issue_date"],
                              f"before suspected unadjusted split on {row['issue_date']} (inconsistent price scale)")

corrupted = detect_adj_factor_anomalies(cleaner.df) & cleaner.df["adj_source"].eq("vendor")
cleaner.repair_adjusted_close(corrupted)

cleaner.validation_log()

,missing_raw_price,nonpositive_or_nonfinite_raw,close_outside_range,open_outside_range,missing_adj_close,nonpositive_or_nonfinite_adj,adj_factor_jump_over_50pct,duplicate_keys,rows
start,0,5,0,5,19855,0,19410,29460,397711
1. duplicates,0,5,0,5,18374,0,19410,0,368251
2. all-missing rows,0,5,0,5,18374,0,19410,0,368251
3. forward-fill prices,0,5,0,5,18374,0,19410,0,368251
4. fill volume,0,5,0,5,18374,0,19410,0,368251
5. impossible prices,0,0,0,0,18374,0,19410,0,368246
6. spike errors,0,0,0,0,18376,0,19408,0,368246
7. MCD before 1968-01-02,0,0,0,0,18360,0,19394,0,367871
8. adjusted close,0,0,0,0,0,0,0,0,367871


The log shows each rule after each step. The five zero opens are on the first day of their ticker's history, with no earlier value to carry forward, so they are quarantined at step 5. The three TRV prints are interpolated at step 6. Step 8 brings missing adjusted closes and adjusted-factor jumps to zero.

In [21]:
spike_view = cleaner.df[cleaner.df["price_interpolated"]][["ticker", "ts", "open", "high", "low", "close", "adj_close", "adj_source"]]
spike_view

,ticker,ts,open,high,low,close,adj_close,adj_source
317952,TRV,1976-10-28,4.650,4.650,4.650,4.650,0.988767,interpolated_factor
318493,TRV,1978-12-20,4.055,4.055,4.055,4.055,0.915262,interpolated_factor
319158,TRV,1981-08-10,6.030,6.030,6.030,6.030,1.570000,interpolated_factor


### Validation

In [22]:
clean = cleaner.df.copy()
quarantine = cleaner.quarantined()

# Rerun the diagnosis on the cleaned data
clean = add_simple_returns(clean)
clean["extreme_return_flag"] = detect_return_outliers(clean)
clean["zero_volume"] = clean["volume"].eq(0)
post = classify_extreme_returns(clean.assign(return_outlier=clean["extreme_return_flag"]))

checks = validate_ohlc(clean)
checks["missing_volume"] = int(clean["volume"].isna().sum())
checks["negative_or_nonfinite_volume"] = int(((clean["volume"] < 0) | ~np.isfinite(clean["volume"])).sum())
checks["spike_or_split_patterns_left"] = int((post["issue_type"] != "extreme_return").sum())
checks["missing_trading_days"] = len(detect_missing_trading_days(clean, calendar))
checks["corrupted_adj_left"] = int(detect_adj_factor_anomalies(clean).sum())
print(checks.to_string())
assert (checks == 0).all(), "cleaned table fails validation"

missing_raw_price               0
nonpositive_or_nonfinite_raw    0
close_outside_range             0
open_outside_range              0
missing_adj_close               0
nonpositive_or_nonfinite_adj    0
adj_factor_jump_over_50pct      0
duplicate_keys                  0
missing_volume                  0
negative_or_nonfinite_volume    0
spike_or_split_patterns_left    0
missing_trading_days            0
corrupted_adj_left              0


In [23]:
# How far the rebuilt adjusted closes are from the supplied reference (expected: equal except the 3 repaired prints)
cmp = clean.merge(reference.rename(columns={"close": "ref_close", "adj_close": "ref_adj"}), on=["ticker", "ts"])
differs = ~np.isclose(cmp["adj_close"], cmp["ref_adj"], rtol=1e-9)
print(f"adj_close differs from reference on {int(differs.sum())} rows:", cmp.loc[differs, ["ticker", "ts"]].values.tolist())
clean["adj_source"].value_counts().to_frame("rows")

adj_close differs from reference on 3 rows: [['TRV', '1976-10-28'], ['TRV', '1978-12-20'], ['TRV', '1981-08-10']]


,rows
adj_source,
vendor,339101
reference_factor,28767
interpolated_factor,3


Row accounting and before/after statistics:

In [24]:
accounting = pd.Series({
    "raw rows": len(raw),
    "exact duplicates removed": cleaner.n_exact_duplicates,
    "quarantined": len(quarantine),
    "retained": len(clean),
})
assert accounting["raw rows"] == accounting.iloc[1:].sum()
print(accounting.to_string())
quarantine.groupby("reason").size().rename("rows").to_frame()

raw rows                    397711
exact duplicates removed     29460
quarantined                    380
retained                    367871


,rows
reason,
before suspected unadjusted split on 1968-01-02 (inconsistent price scale),375
"impossible price, no valid replacement within the day's range",5


In [25]:
stats_cols = ["open", "high", "low", "close", "adj_close", "volume"]
pd.concat({"before": raw[stats_cols].describe().T, "after": clean[stats_cols].describe().T}, axis=1).round(2)

before                                                                                         after                                   \
              count         mean           std   min         25%         50%          75%           max     count         mean           std   min   
open       397711.0        46.29  6.713000e+01  0.00        4.21       19.85        57.53  6.042600e+02  367871.0        46.36  6.716000e+01  0.01   
high       397711.0        46.75  6.778000e+01  0.01        4.24       20.06        58.13  6.086300e+02  367871.0        46.82  6.781000e+01  0.01   
low        397711.0        45.82  6.648000e+01  0.00        4.16       19.63        56.95  5.978800e+02  367871.0        45.89  6.651000e+01  0.00   
close      397711.0        46.29  6.714000e+01  0.01        4.20       19.86        57.53  6.054000e+02  367871.0        46.36  6.717000e+01  0.01   
adj_close  377856.0        51.65  1.945800e+02  0.00        1.39       11.96        43.77  1.044251e+04  367871.0        37.31  6.361000e+01  0.00   
volume     397711.0  30977733.69  1.266237e+08  0.00  1730700.00  4648200.00  11023796.50  9.276606e+09  367871.0  30982504.45  1.266761e+08  0.00   

                                                              
                  25%         50%          75%           max  
open             4.23       19.95        57.62  6.042600e+02  
high             4.27       20.17        58.22  6.086300e+02  
low              4.19       19.73        57.04  5.978800e+02  
close            4.23       19.96        57.63  6.054000e+02  
adj_close        1.32       11.50        41.63  6.054000e+02  
volume     1738800.50  4657885.00  11033922.50  9.276606e+09

### Save

In [26]:
clean_out = clean[["ts", "ticker", "open", "high", "low", "close", "adj_close", "volume",
                   "price_filled", "price_interpolated", "volume_filled", "zero_volume",
                   "extreme_return_flag", "adj_source"]].copy()
clean_out["ts"] = pd.to_datetime(clean_out["ts"]).dt.strftime("%Y-%m-%d")  # ISO dates for SQL filters
assert not clean_out.duplicated(["ticker", "ts"]).any()

clean_out.to_sql(CLEAN_TABLE, conn, if_exists="replace", index=False)
quarantine.drop(columns=["ret"], errors="ignore").to_sql(QUARANTINE_TABLE, conn, if_exists="replace", index=False)
quality_report.to_sql(REPORT_TABLE, conn, if_exists="replace", index=False)
ticker_audit.to_sql(AUDIT_TABLE, conn, if_exists="replace", index=False)
conn.execute(f"CREATE INDEX IF NOT EXISTS idx_{CLEAN_TABLE}_ticker_ts ON {CLEAN_TABLE} (ticker, ts)")
conn.commit()

saved = pd.read_sql_query(f"SELECT COUNT(*) AS n, MIN(ts) AS first, MAX(ts) AS last, COUNT(DISTINCT ticker) AS tickers FROM {CLEAN_TABLE}", conn)
raw_after = pd.read_sql_query(f"SELECT * FROM {RAW_TABLE}", conn)
assert (len(raw_after), pd.util.hash_pandas_object(raw_after, index=False).sum()) == raw_fingerprint, "raw table changed"
print("Raw table unchanged.")
saved

Raw table unchanged.


,n,first,last,tickers
0,367871,1962-01-02,2024-12-06,30
